# Two districts. One banking crisis.

**A guided investigation · Chapter 5 · 30–45 minutes · No Python experience needed**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/intro2causal/blob/main/notebooks_colab/explore-banks_clean.ipynb)

> **The question**  
> Did different Federal Reserve responses help banks survive the Great Depression?

During the Great Depression, Mississippi straddled two Federal Reserve districts that responded differently to banking distress. Richardson and Troost studied this boundary to investigate whether emergency lending helped banks survive. This teaching extract gives six July 1 snapshots of the number of banks still in business in each district. These are operating-bank counts, not counts of failures, and the districts did not start with the same number of banks. The file also supplies a constructed path for the sixth district based on the eighth district’s changes from 1930. That path is a counterfactual illustration, not another observed district. By looking first at levels and then at changes, we can ask why a before-and-after comparison might be insufficient and what a comparison district contributes to the argument. Six observations are enough for an important question, but not for unlimited historical conclusions.

### By the end, you will be able to…

- Compare changes in operating banks and separate observed history from a constructed counterfactual.
- Read and modify three interactive charts using short Python examples.
- Explain one reason a descriptive pattern alone cannot settle this case’s causal question.

### Before you see the evidence

If both districts lose banks, could lending still have helped one district?

Write two sentences now. You will return to them at the end.

**Your route:** meet the data → inspect three views → modify and try → ask the causal question.

## Your first few minutes in Colab · 3 minutes

You do not need prior Python experience. A **cell** is one block of text or code. Read the text, then press the **▶** beside the next code cell (or Shift+Enter). Its output appears underneath. Run cells in order: later cells use tables created earlier.

Choose **File → Save a copy in Drive** if you want to keep your changes. If a name is “not defined,” run the earlier setup cells again. **Runtime → Restart session and run all** gives you a fresh start. An error does not mean you have damaged the data: this notebook only reads it.

Keep a short learning log in a text cell. Before each chart, make a prediction. Afterward, write one observation with a number and one question that the chart cannot answer.

**Chart controls:** hover for values, click legend labels to hide/show a group, drag to zoom, and double-click to reset. A chart’s menu also offers a reset/autoscale control.

The setup installs Plotly Express if needed. `import` makes a library available; `pd` and `px` are short names for pandas (tables) and Plotly Express (charts).

In [ ]:
%pip install -q "plotly>=6,<7"

In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display

# One consistent chart style; no plotting experience needed.
px.defaults.template = "plotly_white"
px.defaults.color_continuous_scale = "Blues"
px.defaults.color_discrete_sequence = ["#2563eb", "#7c3aed", "#059669", "#64748b"]

## Meet the data · 5 minutes

**One row:** Annual July 1 snapshot of banks in business in each Federal Reserve district.  
**Stored size:** 6 rows × 4 columns.

| Column | What it records |
|---|---|
| `year` | Calendar year of the observation; census year in the child-labor file. |
| `bib6` | Number of banks in business in the sixth Federal Reserve district on the July 1 snapshot. |
| `bib8` | Number of banks in business in the eighth Federal Reserve district on the July 1 snapshot. |
| `counterfactual` | Constructed sixth-district path holding the 1930 sixth–eighth district gap fixed; not observed. |

There are only six annual snapshots. bib6 and bib8 count banks in business. The supplied counterfactual holds the 1930 district gap fixed; it is not observed evidence of what would certainly have happened.

`read_csv` downloads a table. We call it `df`. `head(5)` shows just five rows, not a separate sample used for the analysis. Square brackets select columns by their names.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/cmg777/intro2causal/main/data/ch5/banks_clean.csv"
df = pd.read_csv(DATA_URL)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")
display(df[['year', 'bib6', 'bib8', 'counterfactual']].head(5))

### Pause: what do the rows mean?

Choose one displayed row. Translate it into a sentence using the variable guide. Which values are categories or identifiers? Which are quantities? A numeric storage type does not necessarily make a code a meaningful measurement.

The next table checks types, missing values, and the number of distinct values for the key columns. `isna()` marks missing cells; `sum()` counts them. Missing is not zero. Ask which comparison would lose a row if one of its required values were missing.

In [ ]:
key_columns = ['year', 'bib6', 'bib8', 'counterfactual']
checks = pd.DataFrame({"storage_type": df[key_columns].dtypes.astype(str),
                       "missing_rows": df[key_columns].isna().sum(),
                       "distinct_values": df[key_columns].nunique()})
display(checks)
print("The charts below omit missing values required for each plotted quantity.")

### A small summary before plotting

`describe()` summarizes numeric columns: `count` counts available values, `mean` is the average, and `50%` is the median. The median splits ordered values into two halves. Averages of identifiers or arbitrary category codes are not useful descriptions, even if Python calculates them.

For category-like key variables, the second table counts values, including missing entries. Check these labels before interpreting a chart.

In [ ]:
numeric_keys = df[key_columns].select_dtypes(include="number").columns
if len(numeric_keys):
    display(df[numeric_keys].describe().round(2))
for column in key_columns:
    if df[column].nunique() <= 8:
        display(df[column].value_counts(dropna=False).rename("Rows").to_frame())

## Read the chart code · 2 minutes

`px.bar`, `px.scatter`, `px.line`, and `px.box` choose a chart type. `x` and `y` name columns for the axes; `color` distinguishes groups or values. `labels` translates column names for readers. `fig.show()` displays the result.

A table transformation and a plot answer different questions: first decide what each plotted row represents, then choose how to display it. Before running each chart, predict what you expect to see.

In [ ]:
# Display labels: run once; you do not need to memorize this mapping.
labels = {'ability_group': 'Ability group',
 'census': 'Census year',
 'group': 'State group',
 'offer': 'Scholarship assignment',
 'plan': 'Assigned insurance plan',
 'state_name': 'Simulated state',
 'year': 'Year'}

## Visual investigation 1 · Begin with observed history

**Predict:** How many banks remained in business in each district?

The data are sorted by year before drawing lines. A line joins annual snapshots; it does not reveal what happened on every intervening day.

In [ ]:
history = df.sort_values("year").melt(id_vars="year", value_vars=["bib6", "bib8"], var_name="district", value_name="banks")
history["district"] = history["district"].map({"bib6": "6th district", "bib8": "8th district"})
fig = px.line(history, x="year", y="banks", color="district", markers=True, labels={"year": "July 1 snapshot", "banks": "Banks in business", "district": "District"}, title="Operating banks during the crisis")
fig.show()

**Explore the chart:** Hover over 1930 and 1931. Compare starting levels and subsequent movement separately.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

Fewer operating banks is a decline in this count. It is not a direct count of failures, and the starting difference makes a simple level comparison uninformative about policy.

</details>

## Visual investigation 2 · Put the starting point on equal footing

**Predict:** How did each district change relative to 1930?

Subtracting the 1930 count within each district gives a change measured in banks. A zero indicates the baseline, not zero banks.

In [ ]:
baseline = history[history["year"] == 1930].set_index("district")["banks"]
history["change"] = history["banks"] - history["district"].map(baseline)
fig = px.line(history, x="year", y="change", color="district", markers=True, labels={"year": "Year", "change": "Change in banks since 1930", "district": "District"}, title="Compare changes, not just levels")
fig.add_hline(y=0, line_dash="dot")
fig.show()

**Explore the chart:** Compare how far each line falls after 1930. Both can fall while one falls less.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

Different declines motivate a differences-in-differences comparison. They do not prove that the two districts would have followed the same path without intervention.

</details>

## Visual investigation 3 · Make the assumption visible

**Predict:** How does the supplied counterfactual compare with the observed sixth district?

The counterfactual is constructed from the eighth district’s movement and the 1930 gap. We draw it with a different line style to distinguish assumption from observation.

In [ ]:
paths = df.melt(id_vars="year", value_vars=["bib6", "counterfactual"], var_name="path", value_name="banks")
paths["path"] = paths["path"].map({"bib6": "Observed 6th district", "counterfactual": "Constructed counterfactual"})
fig = px.line(paths, x="year", y="banks", color="path", line_dash="path", markers=True, labels={"year": "Year", "banks": "Banks in business", "path": "Path"}, title="Observed and assumed paths")
fig.show()

**Explore the chart:** Toggle the counterfactual off, then on. Which line is observed? Which depends on an assumption?

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

The gap illustrates the question the design asks. Calling it a policy effect requires historical support for the comparison, not just arithmetic.

</details>

## Make it yours · 8 minutes

### 1. Modify an example

Change the baseline year in the change chart to 1929. Explain why the displayed changes move.

Copy that chart cell before editing, so you can compare the original and modified outputs. Say what changed in your question as well as in your code.

### 2. Try a small investigation

Calculate and plot observed sixth-district banks minus the supplied counterfactual for each year.

Write a prediction first. The optional starter cell below is safe to leave unchanged; later cells do not depend on your answer.

In [ ]:
# Your investigation goes here.
# Describe your expected pattern before writing code.

<details>
<summary>Hint 1 · choose the right evidence</summary>

Create a column with df["bib6"] - df["counterfactual"].

</details>

<details>
<summary>Hint 2 · a Python starting point</summary>

Use a bar plot with year on x and the new gap on y.

</details>

<details>
<summary>Optional worked solution · open after trying</summary>

<pre><code>gaps = df.assign(gap=df[&quot;bib6&quot;] - df[&quot;counterfactual&quot;])
fig = px.bar(gaps, x=&quot;year&quot;, y=&quot;gap&quot;, labels={&quot;year&quot;: &quot;Year&quot;, &quot;gap&quot;: &quot;Observed minus counterfactual (banks)&quot;})
fig.show()</code></pre>

The difference is relative to the supplied assumed path. It is not a count directly observed as banks saved by lending.

Copy the code into the starter cell to run it.

</details>

## What can you now say? · 4 minutes

Return to your initial prediction. Write a four-sentence evidence note:

1. **I expected…** State your prediction.
2. **I observed…** Use a chart, a comparison, and the correct units.
3. **This does not establish…** Name a causal claim the evidence cannot settle.
4. **To learn more, I would need…** Propose an additional measurement or a more credible comparison.

### Carry this question into the chapter

**What would make the eighth district a credible guide to how the sixth would have changed without intervention?**

[Continue with Chapter 5: Differences-in-Differences](https://cmg777.github.io/intro2causal/book/_book/notebooks_quarto/05-differences-in-differences.html#a-mississippi-experiment) · [Return to the case collection](https://cmg777.github.io/intro2causal/data-dictionary.html#ch5-banks-clean-csv)

### Source and preparation notes

[Richardson and Troost (2009), Monetary Intervention Mitigated Banking Panics during the Great Depression](https://www.economics.uci.edu/files/docs/faculty_review/richardson_troost_2009_jpe.pdf)

Correction: despite the daily `banks.csv` source, this six-row file is produced by selecting July 1 rows, then holding the 1930 district gap constant to construct `counterfactual`. It is not a daily panel or a full annual aggregation.

These exploratory charts are based on the committed CSV. They are not a complete replication of the original study. Keep the source study, this prepared extract, and your descriptive results distinct.